<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/humanoid_tennis_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Humanoid Tennis Curriculum Training

Train one fixed physical Unitree G1 tennis curriculum stage with Stable-Baselines3, inspect the run artifacts, replay the best checkpoint, and evaluate it on the canonical held-out promotion suite.

Start with **Stage 0: anchored racket intercept**. Stages 1–2 are experimental follow-on tasks. This notebook never changes stage automatically: each run owns one immutable environment configuration.

What this notebook does **not** claim:

- It does not demonstrate two free-standing humanoids learning full tennis.
- It does not implement automatic checkpoint transfer, replay-buffer migration, prior-level rehearsal, or curriculum advancement.
- Stages 3–5 remain unavailable; Stage 6 is an environment-correctness mode without a validated training recipe.
- A quick test validates the pipeline, not policy success. PPO remains an experimental baseline.

## 1. Install

Install the training and notebook extras from the repository. During pull-request review, replace `BRANCH-NAME` with the branch you want to exercise.

In [ ]:
!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics"
# To test an unmerged branch instead:
# !pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics@BRANCH-NAME"


## 2. Configure Colab rendering

Run this before importing the environments so MuJoCo selects EGL correctly. In Colab, choose a GPU runtime first. The helper is a no-op outside Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab

setup_colab()


## 3. Choose one fixed curriculum stage

`STAGE = 0` is the recommended first learning task. The recipes preserve the full 58-value centralized action API, but Stage 0 exposes only two active shoulder controls and Stages 1–2 expose the seven right-arm controls for the current returner.

PPO is the recipe default. SAC may be selected explicitly, but its entropy tuner sees the 51–56 inactive coordinates and is not mask-aware. `QUICK_TEST=True` shortens the run solely to verify installation, callbacks, artifacts, and rendering.

In [ ]:
STAGE = 0  # 0: intercept, 1: anchored return, 2: randomized return

STAGE_RECIPES = {
    0: "HumanoidTennisStage0Intercept",
    1: "HumanoidTennisStage1AnchoredReturn",
    2: "HumanoidTennisStage2RandomizedReturn",
}
if STAGE not in STAGE_RECIPES:
    raise ValueError("This notebook supports only implemented training stages 0, 1, and 2.")

ALGO = None  # None preserves the recipe default (PPO); or set "PPO" / "SAC"
USE_DRIVE = True
QUICK_TEST = False
SEED = 0
TOTAL_TIMESTEPS = None  # None preserves the selected recipe budget
N_ENVS = None  # None preserves the recipe's validated n_envs=1
EARLY_STOP_PATIENCE = 20
MODEL_KWARGS = {}
RUN_ORACLE_PREFLIGHT = True
START_TENSORBOARD = False
RUN_HELD_OUT_EVAL = True
DISCONNECT_WHEN_DONE = False


## 4. Mount Drive and create a run directory

With Drive enabled, checkpoints and reports survive a Colab runtime restart. Every invocation creates a timestamped directory so separate stages and seeds never mix artifacts.

In [ ]:
from courtside_dynamics.notebook_utils import mount_drive, resolve_run_dir
from courtside_dynamics.recipes import RECIPES

if USE_DRIVE:
    mount_drive()

ENV = STAGE_RECIPES[STAGE]
recipe = RECIPES[ENV]
ALGO = ALGO or recipe.default_algo
LOG_DIR = resolve_run_dir(ENV, ALGO, use_drive=USE_DRIVE)
print(f"Stage {STAGE}: {recipe.description}")
print("Logging to:", LOG_DIR)


## 5. Build and inspect the training configuration

The recipe owns the canonical stage horizon, PPO default, one-worker setting, evaluation cadence, checkpoint cadence, recording schema, and success metric. This cell preserves those values unless an override above is explicit. At least one standard evaluation must run so `best_model.zip` and its matching normalizer exist.

In [ ]:
from courtside_dynamics.recipes import build_train_config

recipe_n_envs = recipe.extra_cfg.get("n_envs")
n_envs = N_ENVS if N_ENVS is not None else recipe_n_envs
if n_envs is None:
    n_envs = 1

cfg = build_train_config(
    ENV,
    algo=ALGO,
    log_dir=LOG_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    quick_test=QUICK_TEST,
    seed=SEED,
    n_envs=n_envs,
    early_stop_patience=EARLY_STOP_PATIENCE,
    model_kwargs=MODEL_KWARGS,
)
if cfg.eval_freq > cfg.total_timesteps:
    raise ValueError("total_timesteps must include at least one evaluation")

probe_env = cfg.env_fn()
try:
    observation, reset_info = probe_env.reset(seed=SEED)
    print(f"action={probe_env.action_space.shape}, observation={observation.shape}")
    print(
        f"active controls={reset_info['active_action_count']}/58, "
        f"episode_len={probe_env.episode_len}, serve_side={reset_info['serve_side_name']}"
    )
    assert reset_info["curriculum_stage"] == STAGE
finally:
    probe_env.close()

print(
    f"algo={cfg.algo}, total_timesteps={cfg.total_timesteps:,}, n_envs={cfg.n_envs}, "
    f"eval_freq={cfg.eval_freq:,}, checkpoint_freq={cfg.checkpoint_freq:,}, "
    f"video_freq={cfg.video_freq:,}, normalize_obs={cfg.normalize_obs}"
)


## 6. Physical oracle preflight

Before spending a training budget, verify that the selected deterministic task is physically achievable on both mirrored court sides. Stages 0 and 1 have sensitive, non-learning feasibility fixtures; Stage 2 intentionally has no robust scripted oracle because its feeds are randomized. Oracle success validates physics integration, not trainability.

In [ ]:
from courtside_dynamics.scripted_policies import (
    run_humanoid_tennis_stage0_oracle,
    run_humanoid_tennis_stage1_oracle,
)

if RUN_ORACLE_PREFLIGHT and STAGE in (0, 1):
    runner = (
        run_humanoid_tennis_stage0_oracle
        if STAGE == 0
        else run_humanoid_tennis_stage1_oracle
    )
    oracle_env = cfg.env_fn()
    try:
        oracle_results = [
            runner(oracle_env, serving_side=side, seed=SEED)
            for side in ("a", "b")
        ]
    finally:
        oracle_env.close()
    for result in oracle_results:
        print(
            f"side={result.serving_side.label}, success={result.stage_success}, "
            f"steps={result.steps}, return={result.total_reward:.3f}"
        )
    if not all(result.stage_success for result in oracle_results):
        raise RuntimeError("Physical oracle preflight failed; do not start training.")
elif STAGE == 2:
    print("Stage 2 uses randomized physical feeds and has no robust scripted oracle.")
else:
    print("Oracle preflight skipped by configuration.")


## 7. Live TensorBoard (optional)

Enable `START_TENSORBOARD` to watch evaluation reward, `eval_info/success_rate`, fault metrics, and PPO optimizer diagnostics during a long run.

In [ ]:
if START_TENSORBOARD:
    import os

    from tensorboard import notebook as tb_notebook

    tb_notebook.start(f'--logdir "{os.path.join(LOG_DIR, "tensorboard")}"')
else:
    print("TensorBoard disabled; set START_TENSORBOARD=True to enable it.")


## 8. Train

`train(cfg)` creates the vectorized training/evaluation environments, applies observation normalization, attaches checkpoint/video/info callbacks, and saves both the best and final policies. The returned object is the final policy; canonical evaluation later reloads the best checkpoint from disk.

In [ ]:
from courtside_dynamics.training import train

final_model = train(cfg)


## 9. Run report and diagnostics

Inspect reward, task-success/fault metrics, and PPO optimizer health together. Mean evaluation return selects `best_model.zip`; the stricter held-out success gate remains separate.

In [ ]:
import os

from courtside_dynamics.notebook_utils import (
    plot_eval_info,
    plot_learning_curve,
    plot_training_health,
    print_stage_summary,
)

print_stage_summary(LOG_DIR)
plot_learning_curve(LOG_DIR, save_path=os.path.join(LOG_DIR, "learning_curve.png"))
plot_eval_info(LOG_DIR, save_path=os.path.join(LOG_DIR, "eval_info.png"))
plot_training_health(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "training_health.png"),
)


## 10. Replay the best checkpoint

Reload the best policy and its matching frozen observation statistics, record one deterministic rollout at the selected stage horizon, and embed it here. This video is diagnostic, not promotion evidence.

In [ ]:
from courtside_dynamics.notebook_utils import display_video, record_best_model_video

video_path = record_best_model_video(
    LOG_DIR,
    cfg.env_fn,
    algo=cfg.algo,
    video_length=cfg.video_length,
)
display_video(video_path)


## 11. Canonical held-out evaluation and advisory promotion report

This is deliberately stricter than ordinary SB3 evaluation. It reloads `best_model.zip` rather than using the final in-memory model, pairs it with `best_vec_normalize.pkl`, and evaluates the unchanged recipe environment on the default 100-condition mirrored suite.

For Stage 1 or 2, the same checkpoint and normalizer are also evaluated on every implemented predecessor to measure retention. The gate requires at least 80% current-stage success, at least 75% prior-stage retention, 100 distinct initial states per stage, and zero unsafe episodes by default. It only writes evidence and never advances the curriculum. A quick-test run will normally fail this gate, which is expected.

In [ ]:
import json
from pathlib import Path

from courtside_dynamics.recipes import make_env_fn
from courtside_dynamics.training.algos import resolve_algo
from courtside_dynamics.training.tennis_curriculum import (
    assess_curriculum_promotion,
    evaluate_curriculum_stage,
)

promotion_report = None
if RUN_HELD_OUT_EVAL:
    run_path = Path(LOG_DIR)
    best_model_path = run_path / "best_model.zip"
    best_normalizer_path = run_path / "best_vec_normalize.pkl"
    if not best_model_path.is_file():
        raise FileNotFoundError("best_model.zip is missing; run at least one evaluation.")
    if cfg.normalize_obs and not best_normalizer_path.is_file():
        raise FileNotFoundError(
            "best_vec_normalize.pkl is required for the normalized best policy."
        )

    best_policy = resolve_algo(cfg.algo).load(str(best_model_path), device="cpu")
    policy_id = f"{run_path.name}:best_model"
    normalization_id = (
        f"{run_path.name}:best_vec_normalize" if cfg.normalize_obs else "raw"
    )
    normalization_path = best_normalizer_path if cfg.normalize_obs else None

    held_out_summaries = {}
    for eval_stage in range(STAGE + 1):
        summary = evaluate_curriculum_stage(
            best_policy,
            make_env_fn(STAGE_RECIPES[eval_stage]),
            policy_id=policy_id,
            policy_artifact_path=best_model_path,
            normalization_id=normalization_id,
            normalization_artifact_path=normalization_path,
        )
        held_out_summaries[eval_stage] = summary
        summary_path = run_path / f"held_out_stage_{eval_stage}.json"
        summary_path.write_text(json.dumps(summary.to_dict(), indent=2) + "\n")
        low, high = summary.success_rate_ci95
        nonzero_faults = {
            item.name: item.rate for item in summary.fault_rates if item.count
        }
        print(
            f"stage={eval_stage} success={summary.success_rate:.1%} "
            f"CI95=[{low:.1%}, {high:.1%}] "
            f"unique_states={summary.unique_initial_state_count} "
            f"faults={nonzero_faults}"
        )

    promotion_report = assess_curriculum_promotion(
        held_out_summaries[STAGE],
        prior_stages=tuple(held_out_summaries[index] for index in range(STAGE)),
    )
    promotion_path = run_path / "promotion_report.json"
    promotion_path.write_text(
        json.dumps(promotion_report.to_dict(), indent=2) + "\n"
    )
    print("Eligible for manual promotion:", promotion_report.eligible_for_manual_promotion)
    for reason in promotion_report.reasons:
        print(" -", reason)
    print("Saved:", promotion_path)
else:
    print("Held-out promotion evaluation skipped by configuration.")


## 12. Interpret the result

An eligible report is evidence that this exact checkpoint met the configured gate on held-out launches. It is not an automatic stage transition. Phase 4 has no checkpoint/normalizer transfer workflow, so preserve the run directory and make any next-stage experiment an explicit, separately reviewed decision. JSON summaries are audit artifacts; because there is no JSON-to-summary loader, cross-stage gating must be recomputed in one runtime from the same checkpoint and normalizer.

## 13. Audit artifacts

Check the standard training outputs before disconnecting. When held-out evaluation ran, also require the promotion report and one summary per evaluated stage.

In [ ]:
from courtside_dynamics.notebook_utils import check_run_artifacts

missing = check_run_artifacts(LOG_DIR)
if RUN_HELD_OUT_EVAL:
    expected_promotion_files = [
        Path(LOG_DIR) / "promotion_report.json",
        *(Path(LOG_DIR) / f"held_out_stage_{stage}.json" for stage in range(STAGE + 1)),
    ]
    missing_promotion = [path for path in expected_promotion_files if not path.is_file()]
    if missing_promotion:
        raise FileNotFoundError(f"Missing promotion artifacts: {missing_promotion}")
    print("All held-out evaluation artifacts present.")


## 14. Disconnect Colab (optional)

Set `DISCONNECT_WHEN_DONE=True` before Run All to release the runtime after every artifact and report has been written.

In [ ]:
if DISCONNECT_WHEN_DONE:
    from courtside_dynamics.notebook_utils import disconnect_runtime

    disconnect_runtime(delay_seconds=30)
else:
    print("Runtime left connected for inspection.")
